# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [9]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [10]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [ ]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


In [ ]:
evaluate(human_pricer, test, size=100)

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [ ]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

In [ ]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
evaluate(neural_network, test)

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [12]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [13]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [14]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [ ]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test)

In [15]:
# The function for gpt-4.1-mini

def gpt_4__1_mini(item):
    response = completion(model="openai/gpt-4.1-mini", messages=messages_for(item))
    return response.choices[0].message.content

In [16]:
gpt_4__1_mini(test[0])

'$180'

In [17]:
evaluate(gpt_4__1_mini, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$59 $134 $30 $50 $50 $170 $24 $155 $1 $20 $313 $29 $15 $29 $39 $7 $11 $5 $40 $39 $54 $36 $10 $125 $82 $203 $5 $5 $151 $65 $50 $15 $90 $55 $65 $119 $60 $41 $66 $2 $165 $55 $23 $105 $70 $0 $22 $8 $70 $22 $25 $105 $75 $20 $67 $14 $8 $50 $82 $3 $116 $43 $31 $30 $229 $30 $60 $295 $5 $144 $17 $8 $170 $4 $15 $21 $126 $2 $2 $6 $0 $0 $15 $79 $2 $10 $68 $56 $30 $21 $3 $20 $5 $20 $2 $108 $1 $7 $90 $325 $20 $3 $12 $19 $26 $132 $13 $370 $29 $99 $39 $86 $19 $58 $54 $80 $0 $5 $54 $47 $19 $511 $20 $14 $50 $20 $5 $61 $59 $94 $7 $113 $20 $15 $65 $0 $55 $10 $38 $42 $6 $50 $0 $12 $74 $33 $20 $60 $15 $8 $6 $114 $22 $20 $1 $29 $21 $44 $100 $5 $11 $17 $43 $0 $240 $2 $351 $25 $5 $5 $10 $2 $170 $2 $67 $29 $12 $23 $39 $7 $54 $15 $150 $109 $20 $18 $63 $7 $20 $12 $15 $9 $5 $19 $55 $10 $10 $30 $6 $1 

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test, size=10, workers=1)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [9]:
def gemini_3__flash(item):
    response = completion(model="gemini/gemini-3-flash-preview", messages=messages_for(item))
    return response.choices[0].message.content

In [10]:
evaluate(gemini_3__flash, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $24 $15 $5 $10 $40 $79 $72 $10 $81 $67 $20 $5 $11 $39 $3 $1 $18 $39 $30 $15 $9 $20 $25 $82 $204 $175 $1 $121 $65 $2 $30 $11 $45 $5 $270 $15 $33 $24 $15 $160 $40 $16 $65 $15 $0 $5 $4 $70 $48 $24 $105 $225 $0 $12 $11 $3 $25 $113 $4 $47 $45 $31 $75 $21 $0 $60 $296 $10 $11 $19 $2 $97 $4 $17 $17 $49 $2 $3 $4 $0 $4 $5 $69 $12 $25 $68 $126 $0 $14 $3 $30 $5 $10 $2 $88 $4 $17 $80 $260 $25 $2 $3 $1 $0 $53 $10 $340 $12 $119 $20 $81 $6 $83 $34 $20 $5 $5 $14 $42 $4 $54 $5 $23 $10 $25 $1 $51 $10 $74 $71 $22 $20 $6 $15 $2 $20 $10 $12 $11 $1 $101 $35 $5 $44 $8 $5 $15 $124 $8 $4 $43 $22 $0 $0 $70 $16 $41 $35 $15 $100 $14 $74 $2 $590 $3 $202 $30 $15 $1 $0 $2 $231 $8 $77 $11 $2 $56 $26 $28 $5 $15 $240 $64 $14 $3 $53 $17 $20 $2 $5 $15 $0 $31 $60 $15 $21 $55 $18 $11 

In [7]:
def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [8]:
evaluate(grok_4__1_fast, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $134 $30 $10 $20 $110 $54 $115 $11 $80 $87 $121 $5 $4 $29 $8 $1 $15 $40 $39 $16 $16 $45 $175 $18 $154 $45 $5 $301 $65 $30 $10 $90 $65 $35 $369 $60 $31 $14 $13 $175 $45 $25 $105 $70 $0 $7 $3 $65 $48 $25 $117 $225 $10 $3 $14 $3 $80 $48 $3 $116 $48 $46 $60 $229 $10 $10 $325 $5 $174 $17 $8 $30 $1 $30 $16 $26 $0 $2 $6 $40 $1 $5 $74 $2 $10 $68 $56 $0 $21 $3 $15 $5 $20 $4 $108 $1 $77 $20 $325 $20 $3 $12 $10 $99 $82 $17 $350 $9 $51 $10 $64 $19 $8 $54 $230 $6 $0 $34 $147 $9 $411 $10 $86 $0 $10 $5 $51 $29 $79 $79 $13 $5 $5 $185 $0 $55 $10 $22 $18 $16 $249 $40 $14 $44 $18 $5 $290 $85 $8 $4 $144 $22 $60 $4 $29 $71 $41 $40 $25 $211 $17 $38 $2 $140 $2 $402 $25 $5 $2 $15 $7 $170 $8 $57 $1 $7 $27 $14 $27 $146 $15 $150 $1 $30 $3 $83 $17 $10 $2 $5 $1 $10 $161 $11 $50 $80 $30 $21 $4 

In [ ]:
# The function for gpt-5.2

def gpt_5__2(item):
    response = completion(model="openai/gpt-5.2", messages=messages_for(item), reasoning_effort='none', seed=42)
    return response.choices[0].message.content


In [21]:
evaluate(gpt_5__2, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $94 $20 $10 $10 $100 $54 $5 $6 $10 $386 $71 $5 $1 $39 $3 $21 $15 $20 $19 $34 $6 $5 $125 $33 $204 $195 $2 $121 $62 $10 $20 $30 $60 $85 $220 $20 $43 $34 $13 $160 $50 $5 $45 $30 $0 $7 $8 $46 $12 $22 $100 $225 $10 $37 $14 $5 $109 $78 $3 $106 $48 $61 $10 $30 $9 $10 $285 $25 $74 $17 $3 $90 $2 $20 $16 $56 $1 $3 $4 $0 $1 $0 $74 $14 $30 $118 $44 $30 $16 $8 $25 $5 $15 $1 $22 $1 $7 $90 $14 $20 $13 $7 $10 $19 $282 $16 $311 $1 $110 $20 $86 $4 $38 $54 $59 $5 $0 $4 $397 $9 $310 $10 $6 $20 $0 $5 $41 $21 $99 $189 $23 $5 $10 $65 $2 $6 $30 $47 $22 $16 $449 $20 $5 $14 $48 $10 $10 $25 $8 $4 $84 $27 $60 $1 $51 $61 $36 $60 $20 $41 $18 $28 $2 $240 $7 $351 $20 $0 $3 $10 $3 $120 $10 $56 $31 $3 $37 $66 $13 $246 $25 $220 $59 $20 $3 $53 $27 $20 $2 $5 $1 $5 $11 $10 $40 $10 $0 $16 $1 